In [110]:
# ============================================================
# STEP 1: Import Required Libraries
# ============================================================

import pandas as pd
import numpy as np

# Train-validation split
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Regression Models
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)

# Evaluation Metric
from sklearn.metrics import mean_squared_error

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

In [111]:
# ============================================================
# STEP 2: Load Dataset
# ============================================================

# Replace with your file path if necessary
df = pd.read_csv("/content/healthcare-dataset-stroke-data.csv")

# Display first five rows
display(df.head())

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [112]:
# ============================================================
# STEP 3.1: Check Dataset Shape
# ============================================================

print("Dataset Shape:")
print(df.shape)

Dataset Shape:
(5110, 12)


In [113]:
# ============================================================
# STEP 3.2: Check Data Types
# ============================================================

print("\nData Types:")
print(df.dtypes)


Data Types:
id                     int64
gender                object
age                  float64
hypertension           int64
heart_disease          int64
ever_married          object
work_type             object
Residence_type        object
avg_glucose_level    float64
bmi                  float64
smoking_status        object
stroke                 int64
dtype: object


In [114]:
# ============================================================
# STEP 3.3: Check Missing Values
# ============================================================

print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64


In [115]:
# ============================================================
# STEP 3.4: Check Duplicate Rows
# ============================================================

duplicates = df.duplicated().sum()

print("\nNumber of Duplicate Rows:")
print(duplicates)


Number of Duplicate Rows:
0


In [116]:
# ============================================================
# STEP 4: Remove ID Column
# ============================================================

id_column = df['id']

df = df.drop(columns=['id'])

print("ID column removed.")


# ============================================================
# STEP 5: Temporarily Remove Stroke Target
# ============================================================

stroke_target = df['stroke']

df = df.drop(columns=['stroke'])

print("Stroke target temporarily removed.")

ID column removed.
Stroke target temporarily removed.


In [117]:
# ============================================================
# STEP 6: Verify Missing Features
# ============================================================

missing_counts = df.isnull().sum()

print("Missing Values Per Feature:")
print(missing_counts[missing_counts > 0])

# ============================================================
# STEP 7: Verify BMI Missing Values
# ============================================================

bmi_missing = df['bmi'].isnull().sum()

print(f"Number of missing BMI values: {bmi_missing}")

Missing Values Per Feature:
bmi    201
dtype: int64
Number of missing BMI values: 201


In [118]:
# ============================================================
# STEP 8: Create Complete BMI Subset
# ============================================================

complete_bmi = df[df['bmi'].notnull()].copy()

print("Complete BMI subset shape:")
print(complete_bmi.shape)


# ============================================================
# STEP 9: Create Missing BMI Subset
# ============================================================

missing_bmi = df[df['bmi'].isnull()].copy()

print("Missing BMI subset shape:")
print(missing_bmi.shape)

Complete BMI subset shape:
(4909, 10)
Missing BMI subset shape:
(201, 10)


In [119]:
# ============================================================
# STEP 10: Define X and y
# ============================================================

y = complete_bmi['bmi']

X = complete_bmi.drop(columns=['bmi'])

print("Feature Matrix Shape:", X.shape)
print("Target Vector Shape:", y.shape)

Feature Matrix Shape: (4909, 9)
Target Vector Shape: (4909,)


In [120]:
# ============================================================
# STEP 11: Train-Validation Split
# ============================================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Set Shape:")
print(X_train.shape)

print("\nValidation Set Shape:")
print(X_valid.shape)

Training Set Shape:
(3927, 9)

Validation Set Shape:
(982, 9)


In [121]:
# ============================================================
# STEP 12.1: Identify Feature Types
# ============================================================

numerical_features = X_train.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=['object']
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['age', 'hypertension', 'heart_disease', 'avg_glucose_level']

Categorical Features:
['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']


In [122]:
# ============================================================
# STEP 12.2: Numerical Transformer
# ============================================================

numeric_transformer = Pipeline(
    steps=[
        ('scaler', StandardScaler())
    ]
)

# ============================================================
# STEP 12.3: Categorical Transformer
# ============================================================

categorical_transformer = Pipeline(
    steps=[
        ('encoder', OneHotEncoder(
            handle_unknown='ignore'
        ))
    ]
)

# ============================================================
# STEP 12.4: Complete Preprocessor
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [123]:

# ============================================================
# STEP 13.1: Mean Imputation Evaluation
# ============================================================

rmse_results = {}


# Predict every validation BMI value using the training mean
mean_value = y_train.mean()

y_pred_mean = np.full(
    shape=len(y_valid),
    fill_value=mean_value
)

# Compute RMSE
rmse_mean = np.sqrt(
    mean_squared_error(y_valid, y_pred_mean)
)

rmse_results['Mean'] = rmse_mean

print("Mean RMSE:", rmse_mean)

Mean RMSE: 8.279686243811046


In [124]:
# ============================================================
# STEP 13.2: Median Imputation Evaluation
# ============================================================

median_value = y_train.median()

y_pred_median = np.full(
    shape=len(y_valid),
    fill_value=median_value
)

rmse_median = np.sqrt(
    mean_squared_error(y_valid, y_pred_median)
)

rmse_results['Median'] = rmse_median

print("Median RMSE:", rmse_median)

Median RMSE: 8.340731407216754


In [125]:
# ============================================================
# STEP 13.3: Define Regression Models
# ============================================================

models = {
    'Linear Regression': LinearRegression(),

    'Ridge Regression': Ridge(
        alpha=1.0,
        random_state=42
    ),

    'Decision Tree': DecisionTreeRegressor(
        random_state=42
    ),

    'Random Forest': RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=100,
        random_state=42
    ),

    'Extra Trees': ExtraTreesRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

In [126]:
# ============================================================
# STEP 13.4: Train and Evaluate Models
# ============================================================

for model_name, model in models.items():

    print(f"\nEvaluating {model_name}...")

    # Create pipeline
    pipeline = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', model)
        ]
    )

    # Train model
    pipeline.fit(X_train, y_train)

    # Predict validation BMI
    y_pred = pipeline.predict(X_valid)

    # Compute RMSE
    rmse = np.sqrt(
        mean_squared_error(y_valid, y_pred)
    )

    # Store result
    rmse_results[model_name] = rmse

    print(f"RMSE: {rmse:.4f}")


Evaluating Linear Regression...
RMSE: 7.3526

Evaluating Ridge Regression...
RMSE: 7.3522

Evaluating Decision Tree...
RMSE: 9.8752

Evaluating Random Forest...
RMSE: 7.4107

Evaluating Gradient Boosting...
RMSE: 7.1975

Evaluating Extra Trees...
RMSE: 7.8465


In [127]:
# ============================================================
# STEP 14: RMSE Comparison Table
# ============================================================

rmse_df = pd.DataFrame(
    rmse_results.items(),
    columns=['Method', 'RMSE']
)

rmse_df = rmse_df.sort_values(
    by='RMSE',
    ascending=True
).reset_index(drop=True)

print("\nRMSE Comparison:")
display(rmse_df)


RMSE Comparison:


,Method,RMSE
0,Gradient Boosting,7.197525
1,Ridge Regression,7.352246
2,Linear Regression,7.352578
3,Random Forest,7.410724
4,Extra Trees,7.846488
5,Mean,8.279686
6,Median,8.340731
7,Decision Tree,9.875241


In [128]:
# ============================================================
# STEP 15: Select Best Method
# ============================================================

best_method = rmse_df.iloc[0]['Method']

best_rmse = rmse_df.iloc[0]['RMSE']

print("Best Imputation Method:")
print(best_method)

print("\nBest RMSE:")
print(best_rmse)


# ============================================================
# Retrieve Best Model Object
# ============================================================

best_model = None

if best_method in models:

    best_model = models[best_method]

    print("\nBest model object saved.")

else:

    print(
        "\nBest method is a statistical imputer "
        "(Mean or Median)."
    )

Best Imputation Method:
Gradient Boosting

Best RMSE:
7.197524521189147

Best model object saved.


In [129]:
# ============================================================
# STEP 16.1: Use All Complete BMI Data
# ============================================================

# BMI becomes the target
y_complete = complete_bmi['bmi']

# Remaining columns become features
X_complete = complete_bmi.drop(columns=['bmi'])

print("Complete Feature Matrix Shape:", X_complete.shape)
print("Complete Target Shape:", y_complete.shape)


# ============================================================
# STEP 16.2: Features for Missing BMI Records
# ============================================================

X_missing = missing_bmi.drop(columns=['bmi'])

print("Missing Feature Matrix Shape:", X_missing.shape)

Complete Feature Matrix Shape: (4909, 9)
Complete Target Shape: (4909,)
Missing Feature Matrix Shape: (201, 9)


In [130]:
# ============================================================
# STEP 16.3: Train Final Imputer
# ============================================================

if best_method in models:

    final_pipeline = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', best_model)
        ]
    )

    final_pipeline.fit(X_complete, y_complete)

    print(f"Final {best_method} model trained.")

else:

    print(f"{best_method} does not require training.")

Final Gradient Boosting model trained.


In [131]:
# ============================================================
# STEP 17.1: Impute Missing BMI Values
# ============================================================

if best_method == 'Mean':

    bmi_predictions = np.full(
        len(X_missing),
        y_complete.mean()
    )

elif best_method == 'Median':

    bmi_predictions = np.full(
        len(X_missing),
        y_complete.median()
    )

else:

    bmi_predictions = final_pipeline.predict(X_missing)

print("BMI values successfully imputed.")

# ============================================================
# STEP 17.2: Replace Missing BMI Values
# ============================================================

missing_bmi['bmi'] = bmi_predictions

print("Missing BMI values replaced.")

BMI values successfully imputed.
Missing BMI values replaced.


In [132]:
# ============================================================
# STEP 17.3: Merge Datasets
# ============================================================

df_imputed = pd.concat(
    [complete_bmi, missing_bmi],
    axis=0
)

print("Datasets combined successfully.")

Datasets combined successfully.


In [141]:
# ============================================================
# STEP 18.1: Restore Original Order
# ============================================================

df_imputed = df_imputed.sort_index()

print("Original order restored.")

# ============================================================
# STEP 18.2: Restore Stroke Column
# ============================================================

df_imputed['stroke'] = stroke_target

print("Stroke target restored.")

# ============================================================
# STEP 19: Restore ID Column
# ============================================================

df_imputed['id'] = id_column

print("ID column restored.")


# ============================================================
# STEP 19.1: Restore Original Column Order
# ============================================================

original_columns = [
    'id',
    'gender',
    'age',
    'hypertension',
    'heart_disease',
    'ever_married',
    'work_type',
    'Residence_type',
    'avg_glucose_level',
    'bmi',
    'smoking_status',
    'stroke'
]

df_imputed = df_imputed[original_columns]

print("Original column order restored.")

Original order restored.
Stroke target restored.
ID column restored.
Original column order restored.


In [142]:
# ============================================================
# STEP 20: Final Missing Value Check
# ============================================================

missing_after = df_imputed.isnull().sum()

print("Remaining Missing Values:")
print(missing_after)

Remaining Missing Values:
id                   0
gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64


In [145]:
# ============================================================
# STEP 21: Imputation Report
# ============================================================

report = {
    'Feature Imputed': 'bmi',

    'Total Dataset Size': len(df_imputed),

    'Original Missing BMI': len(missing_bmi),

    'Imputation Method': best_method,

    'Validation RMSE': round(best_rmse, 4),

}

report_df = pd.DataFrame(
    report.items(),
    columns=['Metric', 'Value']
)

print("\nImputation Report")
display(report_df)


Imputation Report


,Metric,Value
0,Feature Imputed,bmi
1,Total Dataset Size,5110
2,Original Missing BMI,201
3,Imputation Method,Gradient Boosting
4,Validation RMSE,7.1975


In [139]:
# ============================================================
# STEP 22: Save Clean Dataset
# ============================================================

df_imputed.to_csv(
    "stroke_dataset_imputed.csv",
    index=False
)

print("Clean dataset saved successfully.")

Clean dataset saved successfully.
